# AMI Job-Postings NLP — v2 (combined + cleaned data)

Runs the **v0.3 AMI signal dictionary** over the combined, cleaned job-postings dataset
to extract key AI / agentic terms — the same signal logic used on the annual reports, so
the two sources stay directly comparable.

This version is built for the **already-cleaned** dataset, which means:
- It **uses the existing `country` and `region` columns** you produced in the cleaning
  step (rather than re-resolving locations from scratch) — so results match the file you
  manually reviewed.
- It adds an **`is_english`** flag per posting (kept, not dropped — you filter later).
- It breaks signal results down by **company**, by **company × country**, and by
  **region (EU / UK / etc.)**.

**How to run:** pick your `ml` kernel (Kernel → Change Kernel), then run top to bottom.
Outputs are written to `ami_job_signals_output_v2/` next to this notebook.

### 0. Environment check

In [1]:
import sys
print("Python executable:", sys.executable)
for pkg in ["pandas", "yaml", "ftfy", "pycountry", "langdetect"]:
    try:
        __import__(pkg)
        print(f"  {pkg:10s} OK")
    except ImportError:
        print(f"  {pkg:10s} MISSING")


Python executable: /opt/homebrew/anaconda3/envs/ml/bin/python
  pandas     OK
  yaml       OK
  ftfy       OK
  pycountry  OK
  langdetect OK


### 1. Install packages (run once if anything is missing above)
If this errors with `externally-managed-environment`, re-run with `%pip install --break-system-packages ...`.

In [2]:
%pip install pandas pyyaml ftfy pycountry langdetect

Note: you may need to restart the kernel to use updated packages.


### 2. Imports

In [2]:
import re
import yaml
from pathlib import Path
from collections import defaultdict, Counter

import pandas as pd

try:
    import ftfy
    _HAS_FTFY = True
except ImportError:
    _HAS_FTFY = False

try:
    from langdetect import detect_langs, DetectorFactory, LangDetectException
    DetectorFactory.seed = 0
    _HAS_LANGDETECT = True
except ImportError:
    _HAS_LANGDETECT = False

print("Imports OK. langdetect:", _HAS_LANGDETECT)


Imports OK. langdetect: True


### 3. Config — edit these

In [3]:
# ---- paths: resolved relative to the repo, not to any one machine ----
def _find_repo_root():
    """Walk up from the working directory until the repo layout is found."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "data").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return here

ROOT        = _find_repo_root()
DATA        = ROOT / "data"
PROCESSED   = ROOT / DATA / "processed"
DICT_PATH   = ROOT / "dictionary" / "ami_signal_dictionary.yaml"
MATCHER_DIR = ROOT / "src"

# Cleaned job-postings corpus. This is the deduplicated, region-tagged file --
# not the raw scrape. See data/README.md.
CSV_PATH = PROCESSED / "job_postings_combined_cleaned.csv"

# Outputs land in data/ so the downstream notebooks find them without configuration.
OUTPUT_DIR = DATA

# Region filter. Empty list = keep everything.
REGION_FILTER = ["EU", "UK"]   # keep only EU/UK postings

# Column names in the cleaned file (already present -- no re-resolution needed).
COL_COMPANY = "company"
COL_TITLE   = "job_title"
COL_DESC    = "full_description"
COL_COUNTRY = "country"     # pre-cleaned canonical country
COL_REGION  = "region"      # pre-tagged EU / UK / NON-EU/UK / UNKNOWN

POSTINGS_AVAILABLE = CSV_PATH.exists()

print(f"Repo root:  {ROOT}")
print(f"Dictionary: {DICT_PATH}  (exists: {DICT_PATH.exists()})")
print(f"Postings:   {CSV_PATH}  (exists: {POSTINGS_AVAILABLE})")
print(f"Outputs ->  {OUTPUT_DIR}")
if not POSTINGS_AVAILABLE:
    print(
        "\n  NOTE: cleaned postings file not found. This notebook cannot run\n"
        "  without it. Its outputs (posting_level_detail.csv, phrase_detail.csv)\n"
        "  are provided in data/, so notebooks 04-06 run regardless."
    )


Repo root:  /Users/savannahwand/Library/CloudStorage/OneDrive-Personal/Documents/Bayes Term 3/GRP/github
Dictionary: /Users/savannahwand/Library/CloudStorage/OneDrive-Personal/Documents/Bayes Term 3/GRP/github/dictionary/ami_signal_dictionary.yaml  (exists: True)
Postings:   /Users/savannahwand/Library/CloudStorage/OneDrive-Personal/Documents/Bayes Term 3/GRP/github/data/processed/job_postings_combined_cleaned.csv  (exists: True)
Outputs ->  /Users/savannahwand/Library/CloudStorage/OneDrive-Personal/Documents/Bayes Term 3/GRP/github/data


### 4. AMI signal dictionary + matcher logic
The agentic gate, sentence splitting, and phrase matching — the same logic used on the annual reports, loading your v0.7 dictionary from the path above.

In [4]:
# =====================================================================
# Local copy of the matcher logic, kept in sync with src/ami_signal_matcher.py.
#
# Negation filtering and ambiguous-term fallthrough ARE applied here, matching
# the report track scored in notebook 01. Section 11 at the end of this
# notebook re-scores everything with an independent implementation of the same
# rules and should report zero differences -- if it reports any, the two have
# drifted apart again.
# =====================================================================

def load_dictionary(path):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def find_terms(text_lower, terms):
    """Return (term, start_char) for each phrase occurrence, longest phrases first."""
    hits = []
    for term in sorted(terms, key=len, reverse=True):
        pattern = r"(?<![a-z0-9])" + re.escape(term) + r"(?![a-z0-9])"
        for m in re.finditer(pattern, text_lower):
            hits.append((term, m.start()))
    return hits

def qualifier_near(text_lower, pos, qualifiers, window_tokens):
    window_chars = window_tokens * 8
    span = text_lower[max(0, pos - window_chars): pos + window_chars]
    return any(q in span for q in qualifiers)

def negated(text_lower, pos, neg_cues, window_tokens):
    window_chars = window_tokens * 8
    span = text_lower[max(0, pos - window_chars):pos]
    return any(c in span for c in neg_cues)

def gated_agentic_hits(sentence_lower, gate, rules, neg_cues):
    """Agentic vs prior-gen automation hits in one sentence, after the gate.

    Negation is applied to all three gate categories, and ambiguous terms with
    no nearby qualifier fall through to the automation denominator rather than
    being dropped. Both behaviours match src/ami_signal_matcher.py.

    neg_cues is a required argument, not a default: passing it explicitly is
    what prevents negation being silently disabled by an empty fallback.
    """
    if not neg_cues:
        raise ValueError(
            "neg_cues is empty -- negation filtering would be silently disabled. "
            "Check the 'negation_cues' key in the dictionary YAML."
        )

    nwin = rules.get("negation_window_tokens", 6)
    qwin = rules.get("qualifier_window_tokens", 12)
    agentic, automation = [], []

    for term, pos in find_terms(sentence_lower, gate["agentic"]):
        if not negated(sentence_lower, pos, neg_cues, nwin):
            agentic.append(term)

    for term, pos in find_terms(sentence_lower, gate["automation_not_agentic"]):
        if not negated(sentence_lower, pos, neg_cues, nwin):
            automation.append(term)

    for term, pos in find_terms(sentence_lower, gate["ambiguous_requires_qualifier"]):
        if negated(sentence_lower, pos, neg_cues, nwin):
            continue
        if qualifier_near(sentence_lower, pos, gate["agentic_qualifiers"], qwin):
            agentic.append(term)
        else:
            automation.append(term)      # fallthrough, not dropped

    return agentic, automation

d = load_dictionary(DICT_PATH)
NEG_CUES = d.get("negation_cues", [])
print("Dictionary loaded:", d["meta"]["name"], "v" + str(d["meta"]["version"]))
print(f"Negation cues: {len(NEG_CUES)}")
if not NEG_CUES:
    raise ValueError("No negation_cues found in the dictionary -- check the YAML key name.")


Dictionary loaded: AMI Signal Dictionary v0.7
Negation cues: 167


### 5. Language detection
Adds `is_english` (yes / no / unknown) per posting. Nothing dropped — kept and flagged.

In [6]:
MIN_CHARS_FOR_LANG = 25
LANG_CONFIDENCE_FLOOR = 0.60

def detect_is_english(text):
    if not _HAS_LANGDETECT or text is None:
        return "unknown"
    t = str(text).strip()
    if len(t) < MIN_CHARS_FOR_LANG:
        return "unknown"
    try:
        langs = detect_langs(t)
    except LangDetectException:
        return "unknown"
    if not langs or langs[0].prob < LANG_CONFIDENCE_FLOOR:
        return "unknown"
    return "yes" if langs[0].lang == "en" else "no"

print("Language detection ready.")


Language detection ready.


### 6. Signal extraction
Runs the full dictionary over one description: the gate, role-evolution, maturity-stage, governance, and tech-stack categories.

In [7]:
def extract_signals(text, d):
    if not text or not str(text).strip():
        return {}
    text_norm = re.sub(r"\s+", " ", str(text).lower())
    gate = d["agentic_gate"]
    rules = d["matching_rules"]
    neg_cues = d.get("negation_cues", [])
    hits = defaultdict(list)

    for sent in split_sentences(text_norm):
        ag, au = gated_agentic_hits(sent, gate, rules, neg_cues)
        if ag: hits["agentic"].extend(ag)
        if au: hits["automation_not_agentic"].extend(au)

    for subcat, terms in d["dimension_2_role_evolution"].items():
        for term, _ in find_terms(text_norm, terms):
            hits[f"role_evolution__{subcat}"].append(term)

    for stage, terms in d["maturity_stage"].items():
        for term, _ in find_terms(text_norm, terms):
            hits[f"maturity__{stage}"].append(term)

    for subcat, terms in d["dimension_4_governance"].items():
        for term, _ in find_terms(text_norm, terms):
            hits[f"governance__{subcat}"].append(term)

    for subcat, terms in d["dimension_3_tech_stack"].items():
        for term, _ in find_terms(text_norm, terms):
            hits[f"tech_stack__{subcat}"].append(term)

    return hits

def summarize(g_postings, g_hits, g_empty, g_phrases, key_names):
    rows = []
    for key in sorted(g_postings):
        n = g_postings[key]
        phrase_ct = Counter()
        for (_, phrase), c in g_phrases[key].items():
            phrase_ct[phrase] += c
        top = ", ".join(f"{p} ({c})" for p, c in phrase_ct.most_common(8))
        row = dict(zip(key_names, key if isinstance(key, tuple) else (key,)))
        row.update({
            "total_job_titles": n,
            "postings_with_no_description": g_empty[key],
            "total_signal_hits": g_hits[key],
            "avg_signals_per_posting": round(g_hits[key]/n, 2) if n else 0,
            "distinct_phrases_found": len(g_phrases[key]),
            "top_phrases": top,
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values("total_signal_hits", ascending=False)

print("Signal extraction ready.")


Signal extraction ready.


### 7. Load data + language flag

In [8]:
# the cleaned file may be utf-8; fall back to latin-1 if a raw re-export sneaks in
try:
    df = pd.read_csv(CSV_PATH)
except UnicodeDecodeError:
    print("utf-8 failed, falling back to latin-1")
    df = pd.read_csv(CSV_PATH, encoding="latin-1")

print(f"Loaded {len(df)} postings, {df[COL_COMPANY].nunique()} companies")

# repair mojibake / encoding corruption in the description text (e.g. "Ã©" -> "é")
# so accented non-English terms match the dictionary correctly
if _HAS_FTFY:
    df[COL_DESC] = df[COL_DESC].apply(lambda t: ftfy.fix_text(str(t)) if pd.notna(t) else t)
    df[COL_TITLE] = df[COL_TITLE].apply(lambda t: ftfy.fix_text(str(t)) if pd.notna(t) else t)
    print("Applied ftfy text repair to descriptions and job titles.")
else:
    print("WARNING: ftfy not installed — accented text may not match. Run the install cell.")

# sanity: confirm the expected pre-cleaned columns exist
for col in [COL_COMPANY, COL_TITLE, COL_DESC, COL_COUNTRY, COL_REGION]:
    assert col in df.columns, f"Expected column '{col}' not found. Columns: {list(df.columns)}"

# language flag
df["is_english"] = df[COL_DESC].apply(detect_is_english)
print("\nLanguage of descriptions:")
print(df["is_english"].value_counts().to_string())

print("\nRegion breakdown:")
print(df[COL_REGION].value_counts().to_string())


Loaded 4073 postings, 29 companies
Applied ftfy text repair to descriptions and job titles.

Language of descriptions:
is_english
yes        2271
no         1791
unknown      11

Region breakdown:
region
EU           2686
NON-EU/UK    1089
UK            298


In [9]:
# optional region filtering (default: keep everything)
if REGION_FILTER:
    before = len(df)
    keep = df[COL_REGION].isin(REGION_FILTER)
    excluded = df.loc[~keep, COL_COMPANY].value_counts()
    df = df.loc[keep].copy()
    print(f"Region filter {REGION_FILTER}: kept {len(df)} of {before} postings.")
    if not excluded.empty:
        print("Excluded by company:")
        for c, n in excluded.items():
            print(f"  {c}: {n}")
else:
    print("No region filter applied — keeping all postings.")


Region filter ['EU', 'UK']: kept 2984 of 4073 postings.
Excluded by company:
  Allianz: 678
  Swiss Re: 162
  Coface: 82
  Mapfre: 50
  Bolttech: 35
  Lemonade: 26
  Hiscox: 24
  Beazley: 18
  Munich Re: 11
  Generali: 3


### 8. Run the matcher

In [10]:
OUTPUT_DIR.mkdir(exist_ok=True)

co_p, co_h, co_e = Counter(), Counter(), Counter()          # by company
co_ph = defaultdict(Counter)
cc_p, cc_h, cc_e = Counter(), Counter(), Counter()          # by company x country
cc_ph = defaultdict(Counter)
rg_p, rg_h, rg_e = Counter(), Counter(), Counter()          # by region
rg_ph = defaultdict(Counter)
posting_rows = []

for _, row in df.iterrows():
    company = row.get(COL_COMPANY, "UNKNOWN")
    if pd.isna(company): company = "UNKNOWN"
    country = row.get(COL_COUNTRY, "UNKNOWN")
    if pd.isna(country): country = "UNKNOWN"
    region = row.get(COL_REGION, "UNKNOWN")
    if pd.isna(region): region = "UNKNOWN"
    title = row.get(COL_TITLE, "")
    desc = row.get(COL_DESC, "")

    co_p[company] += 1
    cc_p[(company, country)] += 1
    rg_p[region] += 1

    if pd.isna(desc) or not str(desc).strip():
        co_e[company] += 1; cc_e[(company, country)] += 1; rg_e[region] += 1
        hits = {}
    else:
        hits = extract_signals(desc, d)

    flat = [p for plist in hits.values() for p in plist]
    co_h[company] += len(flat); cc_h[(company, country)] += len(flat); rg_h[region] += len(flat)

    for cat, plist in hits.items():
        for p in plist:
            co_ph[company][(cat, p)] += 1
            cc_ph[(company, country)][(cat, p)] += 1
            rg_ph[region][(cat, p)] += 1

    posting_rows.append({
        "company": company, "country": country, "region": region,
        "is_english": row.get("is_english", "unknown"),
        "job_title": title,
        "signal_count": len(flat),
        "distinct_phrases": len(set(flat)),
        "phrases_found": "; ".join(sorted(set(flat))),
    })

company_summary = summarize(co_p, co_h, co_e, co_ph, ["company"])
company_country_summary = summarize(cc_p, cc_h, cc_e, cc_ph, ["company", "country"])
region_summary = summarize(rg_p, rg_h, rg_e, rg_ph, ["region"])

company_summary.to_csv(OUTPUT_DIR / "company_summary.csv", index=False)
company_country_summary.to_csv(OUTPUT_DIR / "company_country_summary.csv", index=False)
region_summary.to_csv(OUTPUT_DIR / "region_summary.csv", index=False)

# long-format phrase detail (company x country x category x phrase)
detail_rows = []
for (company, country), counter in cc_ph.items():
    for (cat, phrase), count in counter.items():
        detail_rows.append({"company": company, "country": country,
                            "category": cat, "phrase": phrase, "count": count})
detail_df = pd.DataFrame(detail_rows)
if not detail_df.empty:
    detail_df = detail_df.sort_values(["company", "country", "category", "count"],
                                      ascending=[True, True, True, False])
detail_df.to_csv(OUTPUT_DIR / "phrase_detail.csv", index=False)

posting_level_df = pd.DataFrame(posting_rows)
posting_level_df.to_csv(OUTPUT_DIR / "posting_level_detail.csv", index=False)

print(f"Done. Outputs written to {OUTPUT_DIR}")


Done. Outputs written to /Users/savannahwand/Library/CloudStorage/OneDrive-Personal/Documents/Bayes Term 3/GRP/github/data


### 9. Results

In [11]:
print("=== By company ===")
company_summary

=== By company ===


,company,total_job_titles,postings_with_no_description,total_signal_hits,avg_signals_per_posting,distinct_phrases_found,top_phrases
3,Allianz,1138,0,6045,5.31,249,"ai (1109), fairness (350), ia (343), intellige..."
9,Generali,245,0,1460,5.96,172,"ai (350), automation (88), copilot (63), ml (4..."
21,Swiss Re,138,0,1287,9.33,124,"ai (365), evaluation (131), data privacy (125)..."
18,NN Group,62,0,629,10.15,85,"ai (105), testing (50), api (43), automation (..."
27,Zego,32,0,475,14.84,49,"ai (218), experiment (32), claude (27), ai ass..."
12,Hiscox,79,0,434,5.49,105,"ai (59), automation (26), end-to-end (19), cop..."
24,Unipol,113,0,321,2.84,41,"ai (222), automazione (11), intelligenza artif..."
22,Talanx,160,0,304,1.90,69,"ai (47), ki (43), copilot (33), integration (2..."
1,Admiral,31,1,226,7.29,57,"integration (29), guidewire (26), model valida..."
26,Wakam,13,0,193,14.85,47,"ai (46), explore (28), ai agents (14), genai (..."


In [12]:
print("=== By region ===")
region_summary

=== By region ===


,region,total_job_titles,postings_with_no_description,total_signal_hits,avg_signals_per_posting,distinct_phrases_found,top_phrases
0,EU,2686,0,11027,4.11,337,"ai (2374), automation (489), ia (433), copilot..."
1,UK,298,1,1859,6.24,159,"ai (385), assess (76), machine learning (70), ..."


In [13]:
print("=== By company x country ===")
company_country_summary.head(40)

=== By company x country ===


,company,country,total_job_titles,postings_with_no_description,total_signal_hits,avg_signals_per_posting,distinct_phrases_found,top_phrases
54,Generali,Italy,235,0,1438,6.12,172,"ai (350), automation (86), copilot (63), ml (4..."
23,Allianz,Spain,147,0,1322,8.99,139,"ai (278), automation (90), end-to-end (84), fa..."
12,Allianz,Germany,228,0,1194,5.24,125,"ai (252), fairness (106), copilot (90), künstl..."
11,Allianz,France,332,0,861,2.59,65,"ia (302), intelligence artificielle (282), ai ..."
94,Swiss Re,Slovakia,80,0,821,10.26,104,"ai (233), evaluation (74), data privacy (72), ..."
21,Allianz,Romania,67,0,727,10.85,78,"ai (108), data privacy (60), artificial intell..."
80,NN Group,Netherlands,62,0,629,10.15,85,"ai (105), testing (50), api (43), automation (..."
25,Allianz,United Kingdom,87,0,489,5.62,74,"artificial intelligence (54), fairness (52), a..."
106,Zego,England,24,0,328,13.67,44,"ai (163), experiment (22), claude (20), ai ass..."
99,Unipol,Italy,113,0,321,2.84,41,"ai (222), automazione (11), intelligenza artif..."


In [14]:
print("=== Phrase detail (sample) ===")
detail_df.head(50)

=== Phrase detail (sample) ===


,company,country,category,phrase,count
1039,AXA,France,agentic,ia,14
1038,AXA,France,agentic,agentique,6
1035,AXA,France,agentic,ia agentique,3
1054,AXA,France,agentic,ia générative,2
1055,AXA,France,agentic,ml,2
1036,AXA,France,agentic,ia embarquée,1
1037,AXA,France,agentic,ia frugale,1
1047,AXA,France,agentic,genai,1
1048,AXA,France,agentic,machine learning,1
1053,AXA,France,agentic,chatbot,1
